# Aircraft Age Overview

What is the age distribution of aircraft observed by the system? Ages are derived at observation time; missing manufacture years remain visible in the coverage table.

In [ ]:
%run ../pathutils.ipynb
%run ../database.ipynb
%run ../export.ipynb
%run ../report-header.ipynb
%run ./aircraft-population-utils.ipynb

export_outputs = True
export_folder = get_export_folder_path()

In [ ]:
report_metadata = display_report_header('Aircraft Age Overview')
population = load_aircraft_population()
aircraft = unique_aircraft(population)
coverage = coverage_summary(population)
coverage

In [ ]:
# Describe known ages separately so unavailable enrichment is never treated as zero.
known = aircraft.dropna(subset=['Age At Observation']).copy()
age_statistics = known['Age At Observation'].describe(percentiles=AGE_PERCENTILES).rename('Value').rename_axis('Statistic').reset_index()
percentiles = known['Age At Observation'].quantile(AGE_PERCENTILES).rename_axis('Percentile').reset_index(name='Age')
oldest_aircraft = known.sort_values('Age At Observation', ascending=False)[['Address', 'Registration', 'Model', 'ICAO Type', 'Manufacturer', 'Manufacture Year', 'Age At Observation', 'First Observation']].head(25)
age_statistics

In [ ]:
# Compare the exploratory threshold with the full continuous distribution.
fig, ax = plt.subplots(figsize=(11, 6))
ax.hist(known['Age At Observation'].to_numpy(dtype=float), bins='auto', edgecolor='white')
ax.set_title('Aircraft age at most recent observation')
ax.axvline(60, color='firebrick', linestyle='--', label='Exploratory 60-year threshold')
ax.set_xlabel('Age (years)')
ax.legend()
if export_outputs:
    export_chart(export_folder, 'aircraft-age-overview', 'png')
    export_to_spreadsheet(export_folder, 'aircraft-age-overview.xlsx', {'Coverage': coverage, 'Statistics': age_statistics, 'Percentiles': percentiles, 'Oldest Aircraft': oldest_aircraft})

In [ ]:
oldest_aircraft